# Partition DMS initial full load

Chạy notebook này **sau khi** `dms/run.ipynb -> status()` báo full load hoàn tất và `errors=0`. Job đọc raw DMS Parquet, partition theo `DATE_COLUMN`, ghi curated Hive style rồi cập nhật Glue Catalog.

## 1. Load config

Notebook dùng chính `dms/.env`. Cần có `CURATED_PREFIX`, `GLUE_DATABASE` và `GLUE_TABLE_PREFIX`.

In [ ]:
%pip install boto3 python-dotenv -q

In [18]:
import importlib.util
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if not (notebook_dir / 'partition_initial.py').is_file():
    notebook_dir = Path.cwd() / 'dms'
module_path = (notebook_dir / 'partition_initial.py').resolve()
sys.modules.pop('partition_initial', None)
spec = importlib.util.spec_from_file_location('partition_initial', module_path)
if spec is None or spec.loader is None:
    raise ImportError(f'Không thể load module từ {module_path}')
partition_initial = importlib.util.module_from_spec(spec)
sys.modules['partition_initial'] = partition_initial
spec.loader.exec_module(partition_initial)
print(f'Loaded code: {partition_initial.__file__}')
assert hasattr(partition_initial, 'retry_crawler'), f'File chưa có retry_crawler: {module_path}'
cfg = partition_initial.config()
print(f'Raw     : s3://{cfg.bucket}/{cfg.raw_prefix}/{cfg.schema}/{cfg.table}/')
print(f'Curated : s3://{cfg.bucket}/{cfg.curated_prefix}/year=YYYY/month=MM/day=DD/')
print(f'Partition column: {cfg.date_column}')

Loaded code: D:\aws\Archiver-Data\dms\partition_initial.py
Raw     : s3://my-data-lake-lklklklkklkiet/raw/rds/orders/public/orders/
Curated : s3://my-data-lake-lklklklkklkiet/curated/rds/orders/year=YYYY/month=MM/day=DD/
Partition column: created_at_utc


## 2. Tạo/update Glue job và crawler

Bước này chưa xử lý dữ liệu; chỉ tạo IAM role, upload Glue script, tạo job và crawler.

In [19]:
partition_initial.setup_partition_job()

Glue job ready: orders-initial-partition-initial
Source : s3://my-data-lake-lklklklkklkiet/raw/rds/orders/public/orders/
Target : s3://my-data-lake-lklklklkklkiet/curated/rds/orders/year=YYYY/month=MM/day=DD/


## 3. Chạy repartition

`wait=False` trả về ngay để notebook không bị giữ lâu. Cell status bên dưới sẽ tự chạy crawler khi Glue job `SUCCEEDED`. Chỉ chạy trước khi bật RDS daily pipeline.

In [3]:
run_id = partition_initial.run_partition_job(wait=False)

C:\Users\sirtu\AppData\Roaming\Python\Python39\site-packages\boto3\compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Started Glue run: jr_f58dfb1703f8a6188a4f98f9badc490cd0107b06f6906c0fd01cb4aeb6569466


In [13]:
partition_initial.status_partition_job(run_id)

Run   : jr_f58dfb1703f8a6188a4f98f9badc490cd0107b06f6906c0fd01cb4aeb6569466
State : RUNNING


## 4. Retry riêng crawler

Nếu Parquet job đã `SUCCEEDED` nhưng crawler lỗi, hàm này repair IAM role và retry riêng crawler; không chạy lại repartition job.

In [20]:
partition_initial.retry_crawler()

Glue job ready: orders-initial-partition-initial
Source : s3://my-data-lake-lklklklkklkiet/raw/rds/orders/public/orders/
Target : s3://my-data-lake-lklklklkklkiet/curated/rds/orders/year=YYYY/month=MM/day=DD/
Using successful Glue run: jr_ddbf937aae63cac3d3c1ba342d365c30abd4126d9084add673aa29f47026f74c
Started crawler: orders-initial-partition-initial-crawler
Crawler state: RUNNING
Crawler state: RUNNING
Crawler state: RUNNING
Crawler state: RUNNING
Crawler state: RUNNING
Crawler succeeded: orders-initial-partition-initial-crawler


## 5. Destroy Glue bootstrap resources

Xóa Glue job, crawler và IAM role. **Giữ nguyên DMS raw, curated S3, Glue database và catalog tables.**

In [21]:
partition_initial.destroy_partition_job()  # Chỉ teardown compute resources.

Deleted crawler: orders-initial-partition-initial-crawler
Deleted Glue job: orders-initial-partition-initial
Deleted IAM role: orders-initial-partition-initial-role
Preserved DMS raw data, curated data, Glue database and catalog tables.
